<a href="https://colab.research.google.com/github/bfs0/SDAIA_Athkax_A-B_Testing_PROJECT/blob/main/Copy_of_%D8%AA%D9%85%D8%B1%D9%8A%D9%86_3_2%D8%8C_%D8%A7%D9%84%D9%85%D8%B7%D8%A7%D8%A8%D9%82%D8%A9_%D9%85%D8%B9_%D9%85%D8%AA%D8%BA%D9%8A%D8%B1_%D9%85%D8%B1%D8%A8%D9%83_%D9%85%D8%AE%D9%81%D9%8A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl">

# تمرين 3.2: المطابقة مع متغير مربك مخفي

**برنامج التجارب واختبارات A/B والاستدلال السببي**، اليوم الثالث

**الخطوات:**
1. من قائمة File اختر Save a copy in Drive، واعمل على النسخة التي تُفتح.
2. شغّل الخلايا بالترتيب، ثم اكتب حلك في الخلية المخصصة.
3. اضغط Share، واجعل General access على Anyone with the link بصلاحية Viewer، ثم اضغط Copy link.
4. الصق الرابط في ملف التقييم الخاص بمجموعتك، في عمود «تمرين 3.2».

</div>

<div dir="rtl">

## الإعداد

شغّل الخلية التالية أولًا.

</div>

In [1]:
import warnings
warnings.filterwarnings("ignore")

import html
from IPython.display import HTML, display

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cycler import cycler
from matplotlib import ft2font
from scipy import stats

# الإصدارات الحديثة من matplotlib تعرض العربية مباشرة، والقديمة تحتاج مكتبتين صغيرتين
NATIVE_ARABIC = hasattr(ft2font, "__libraqm_version__")
if not NATIVE_ARABIC:
    try:
        import arabic_reshaper
        from bidi.algorithm import get_display
    except ImportError:
        import subprocess
        import sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "arabic-reshaper", "python-bidi"], check=True)
        import arabic_reshaper
        from bidi.algorithm import get_display

rng = np.random.default_rng(2026)

# ألوان البرنامج: A تركوازي، وB برتقالي، والقيمة الحقيقية كحلي، والطريقة الساذجة رمادي
TEAL, ORANGE, NAVY, GRAY = "#0E7490", "#E58A1F", "#0F2A3F", "#A9B4BC"
plt.rcParams.update({
    "figure.figsize": (9, 4.5), "figure.dpi": 110,
    "axes.prop_cycle": cycler(color=[TEAL, ORANGE, NAVY, GRAY]),
    "axes.edgecolor": "#56656F", "axes.labelcolor": NAVY, "text.color": NAVY,
    "xtick.color": "#3D4F5C", "ytick.color": "#3D4F5C",
    "axes.grid": True, "grid.color": "#D8E1E7", "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 13, "axes.titleweight": "bold", "font.size": 11,
    "legend.frameon": False,
})
pd.set_option("display.precision", 4)


def ar(text):
    """يجهز النص العربي ليظهر بالشكل والاتجاه الصحيحين داخل الرسوم."""
    text = str(text)
    return text if NATIVE_ARABIC else get_display(arabic_reshaper.reshape(text))


def say(*parts, sep=" "):
    """يطبع نصًا عربيًا من اليمين إلى اليسار."""
    text = html.escape(sep.join(str(p) for p in parts))
    display(HTML(f'<div dir="rtl" style="white-space: pre-wrap; line-height: 1.7;">{text}</div>'))

import statsmodels.formula.api as smf
from sklearn.neighbors import NearestNeighbors


def tidy(fit, terms=None):
    """جدول مختصر لنتيجة الانحدار."""
    ci = fit.conf_int()
    out = pd.DataFrame({"المعامل": fit.params, "SE": fit.bse, "أدنى 95%": ci[0], "أعلى 95%": ci[1], "p": fit.pvalues})
    return out.loc[terms] if terms else out


def smd(x, group):
    """الفرق المعياري بين متوسطي مجموعتين."""
    x, group = np.asarray(x, dtype=float), np.asarray(group)
    a, b = x[group == 1], x[group == 0]
    return (a.mean() - b.mean()) / np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2)


say("الإعداد جاهز")

<div dir="rtl">

## الأدوات الجاهزة

هذه الدوال والبيانات من دفتر اليوم، ويحتاجها التمرين. شغّلها كما هي.

</div>

In [6]:
def simulate_training(n=4_000, seed=6, hidden_strength=0.0):
    r = np.random.default_rng(seed)
    study_hours = r.gamma(3, 2, n)                               # ساعات المذاكرة في الأسبوع
    prior_score = r.normal(60, 10, n)                            # الدرجة السابقة
    motivation = r.normal(0, 1, n)                               # الحماس (لا نقيسه)
    logit = -5.3 + 0.25 * study_hours + 0.05 * prior_score + hidden_strength * motivation
    joined = r.binomial(1, 1 / (1 + np.exp(-logit)))
    outcome = (20 + 1.5 * study_hours + 0.6 * prior_score + 5.0 * joined
               + 4 * hidden_strength * motivation + r.normal(0, 8, n))   # درجة الاختبار النهائي
    return pd.DataFrame({"study_hours": study_hours, "prior_score": prior_score, "joined": joined, "outcome": outcome})


tr = simulate_training()
naive = tr.loc[tr.joined == 1, "outcome"].mean() - tr.loc[tr.joined == 0, "outcome"].mean()
say(f"نسبة من حضروا: {tr.joined.mean():.0%}، والفرق الساذج: {naive:.2f} درجة، والأثر الحقيقي: 5.00 درجات")

<div dir="rtl">

## المطلوب

شغّل المحاكاة مع **متغير مربك مخفي**: `simulate_training(hidden_strength=1.0)`، ثم طبق المطابقة على المتغيرين نفسيهما. هل عاد التقدير قريبًا من 5؟ لماذا؟

</div>

In [11]:
tr_h = simulate_training(hidden_strength=1.0, seed=6)
naive_h = tr_h.loc[tr_h.joined == 1, "outcome"].mean() - tr_h.loc[tr_h.joined == 0, "outcome"].mean()
say(f"نسبة من حضروا: {tr_h.joined.mean():.0%}، والفرق الساذج: {naive_h:.2f} درجات | الأثر الحقيقي: 5.00 درجات")

tr_h["ps"] = smf.logit("joined ~ study_hours + prior_score", data=tr_h).fit(disp=0).predict(tr_h)

treated_h = tr_h[tr_h.joined == 1]
control_h = tr_h[tr_h.joined == 0]
nearest = np.abs(treated_h["ps"].values[:, None] - control_h["ps"].values[None, :]).argmin(axis=1)
matched_h = pd.concat([treated_h, control_h.iloc[nearest]])

att_h = treated_h["outcome"].mean() - control_h["outcome"].iloc[nearest].mean()
say(f"الفرق الساذج: {naive_h:.2f} | بعد المطابقة: {att_h:.2f} | الأثر الحقيقي: 5.00")

pd.DataFrame({"قبل المطابقة": [smd(tr_h[c], tr_h.joined) for c in ["study_hours", "prior_score"]],
              "بعد المطابقة": [smd(matched_h[c], matched_h.joined) for c in ["study_hours", "prior_score"]]},
             index=["study_hours", "prior_score"]).round(3)


# هل عاد التقدير قريباً من 5؟
# لا، لم يعد قريباً من 5 (توقف عند 8.03).
# لماذا؟
# لأننا قمنا بالمطابقة على المتغيرات الظاهرة فقط، وبقي المتغير المخفي (الحماس) يؤثر على النتيجة ويسبب تحيزاً لا تعالجه المطابقة.

,قبل المطابقة,بعد المطابقة
study_hours,0.693,0.007
prior_score,0.349,-0.002


<div dir="rtl">

## التسليم

انتهيت؟ اضغط Share، ثم Copy link، والصق الرابط في ملف التقييم في عمود «تمرين 3.2».

</div>